# 04 · Team and season keys

**Goal:** give every raw team string a *franchise* and every season label a *calendar year*, add them to `matches` and `innings` as key columns, and prove the keys with structural checks.

**Rules:** keys are only ever **added**, next to the raw columns — nothing raw is overwritten, so every earlier notebook stays valid. The franchise identity rule is written down *before* any count is looked at, and each piece of evidence is marked as coming from **the data** or from **outside knowledge**. No fuzzy matching: the reference tables are explicit and an unknown string raises. Venues and cities are the next session.

**Contents:** Part B franchises · Part C seasons · Part D apply the keys · what this changes in the earlier worklists

## Part B — Franchises

### B1 · Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from ipl_eda import info_files, reference, tables

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Works whether Jupyter starts in notebooks/ (the default) or in the repo root.
RAW = next((p for p in [Path("../data/procured/cricsheet"), Path("data/procured/cricsheet")] if p.exists()), None)
if RAW is None:
    raise FileNotFoundError("data/procured/cricsheet not found - see README.md")
PROCESSED = RAW.parents[1] / "processed"
REFERENCE = RAW.parents[1] / "reference"

deliveries = pd.read_csv(RAW / "all_matches.csv", usecols=["match_id", "batting_team", "bowling_team"], low_memory=False)
innings_table = tables.read_innings(PROCESSED / "innings.csv")
matches = tables.read_matches(PROCESSED / "matches.csv")
print(f"deliveries: {len(deliveries):,} rows | innings: {len(innings_table):,} rows | matches: {len(matches):,} rows")

deliveries: 295,732 rows | innings: 2,514 rows | matches: 1,243 rows


### B2 · Every team string, from every column that holds one

In [2]:
sources = {
    "deliveries.batting_team": deliveries["batting_team"], "deliveries.bowling_team": deliveries["bowling_team"],
    "innings.batting_team": innings_table["batting_team"], "innings.bowling_team": innings_table["bowling_team"],
    "matches.team1": matches["team1"], "matches.team2": matches["team2"],
    "matches.toss_winner": matches["toss_winner"], "matches.winner": matches["winner"], "matches.eliminator": matches["eliminator"],
}
names = {source: set(values.dropna()) for source, values in sources.items()}
all_names = sorted(set().union(*names.values()))
print(f"distinct team strings over all nine columns: {len(all_names)}")
pd.DataFrame({
    "distinct strings": {source: len(found) for source, found in names.items()},
    "is the full set of 19": {source: found == set(all_names) for source, found in names.items()},
    "strings it lacks": {source: len(set(all_names) - found) for source, found in names.items()},
})

distinct team strings over all nine columns: 19


,distinct strings,is the full set of 19,strings it lacks
deliveries.batting_team,19,True,0
deliveries.bowling_team,19,True,0
innings.batting_team,19,True,0
innings.bowling_team,19,True,0
matches.team1,19,True,0
matches.team2,19,True,0
matches.toss_winner,19,True,0
matches.winner,19,True,0
matches.eliminator,7,False,12


**One set of 19.** Eight of the nine columns hold all 19 strings; only `eliminator` holds fewer (7), because only seven teams ever won a super over. No column holds a string the others lack, so one reference table can cover every table.

### B3 · The identity rule (written before any counts)

**What makes two raw names one franchise?** One franchise is one continuing club: the same side under a new name or spelling. Two names are **one** franchise (a *rename*) when the later name continues the earlier club. They are **two** franchises (a *replacement*) when the earlier club ended and a new club took its place — even in the same city, even if some players carried over. A club that sat out seasons under one name (a suspension) is still one franchise. A *spelling variant* — a one-character or suffix difference in consecutive seasons — is a rename of the weakest kind.

**What the data can show:** a club has one name at a time, so two names of one franchise never share a season; and a rename follows straight on, so the new name's first season is the season after the old name's last. Both are *necessary* for a rename and *not sufficient*: a replacement looks the same. What separates a rename from a replacement is whether the club itself continued (owners, licence), and that is **outside knowledge**.

An **ambiguous pair** is two names that never share a season and abut in time (the first ends in the season before the second begins). Each is decided on its own below, with the data evidence and the outside knowledge marked separately. Two names with a gap of one or more seasons between them are not ambiguous: they are different franchises.

### B4 · Which pairs are ambiguous? (spans computed, not typed)

In [3]:
team_seasons = pd.concat([
    matches[["match_id", "start_date", "team1"]].rename(columns={"team1": "team"}),
    matches[["match_id", "start_date", "team2"]].rename(columns={"team2": "team"}),
])
team_seasons["year"] = team_seasons["start_date"].dt.year       # the calendar year of the match; part C derives the same year per season label

spans = (team_seasons.groupby("team")
         .agg(first_year=("year", "min"), last_year=("year", "max"), seasons=("year", "nunique"), matches=("match_id", "nunique"))
         .sort_values(["first_year", "team"]))
spans

,first_year,last_year,seasons,matches
team,,,,
Chennai Super Kings,2008,2026,17,266
Deccan Chargers,2008,2012,5,75
Delhi Daredevils,2008,2018,11,161
Kings XI Punjab,2008,2020,13,190
Kolkata Knight Riders,2008,2026,19,278
Mumbai Indians,2008,2026,19,291
Rajasthan Royals,2008,2026,17,251
Royal Challengers Bangalore,2008,2023,16,240
Kochi Tuskers Kerala,2011,2011,1,14


In [4]:
years_of = team_seasons.groupby("team")["year"].agg(set)
pairs = [
    {"earlier name": a, "later name": b, "earlier ends": spans.loc[a, "last_year"], "later begins": spans.loc[b, "first_year"],
     "ever share a season": bool(years_of[a] & years_of[b])}
    for a in spans.index for b in spans.index
    if a != b and spans.loc[a, "last_year"] + 1 == spans.loc[b, "first_year"]
]
ambiguous = pd.DataFrame(pairs)
assert not ambiguous["ever share a season"].any()
ambiguous

,earlier name,later name,earlier ends,later begins,ever share a season
0,Deccan Chargers,Sunrisers Hyderabad,2012,2013,False
1,Delhi Daredevils,Delhi Capitals,2018,2019,False
2,Kings XI Punjab,Punjab Kings,2020,2021,False
3,Royal Challengers Bangalore,Royal Challengers Bengaluru,2023,2024,False
4,Rising Pune Supergiants,Rising Pune Supergiant,2016,2017,False


Five pairs abut without sharing a season. Two more are *not* ambiguous by the rule — **Pune Warriors** (2011–2013) / **Rising Pune Supergiant(s)** (2016–2017) and **Gujarat Lions** (2016–2017) / **Gujarat Titans** (2022 on) are separated by gaps of two and four seasons — and **Kochi Tuskers Kerala** has a single season.

### B5 · Could anything in the data tell a rename from a replacement?

The data evidence available is squad continuity: the `player` lines of the info files say who played for each team in each season. A renamed club should keep much of its squad; a replacement, one would expect, much less. The test: for each ambiguous pair, the share of the earlier name's last-season players who also appear for the later name in its first season — against what the data show for the same name in consecutive seasons, and for names that are certainly different clubs.

In [5]:
long, _ = info_files.read_info_folder(RAW)
match_year = matches.set_index("match_id")["start_date"].dt.year.rename("year")
players = (long[long["key"] == "player"][["match_id", "value1", "value2"]]
           .rename(columns={"value1": "team", "value2": "player"}).join(match_year, on="match_id"))
squad = players.groupby(["team", "year"])["player"].agg(set)

def overlap(earlier, earlier_year, later, later_year):
    """Share of the earlier squad that appears in the later squad."""
    return len(squad[(earlier, earlier_year)] & squad[(later, later_year)]) / len(squad[(earlier, earlier_year)])

ambiguous["squad overlap"] = [overlap(r["earlier name"], r["earlier ends"], r["later name"], r["later begins"]) for _, r in ambiguous.iterrows()]

ambiguous_pairs = set(zip(ambiguous["earlier name"], ambiguous["later name"]))
same_name, different_clubs = [], []
for (team, year), earlier in squad.items():
    for (other, other_year), later in squad.items():
        if other_year == year + 1:
            share = len(earlier & later) / len(earlier)
            if other == team:
                same_name.append(share)
            elif (team, other) not in ambiguous_pairs:
                different_clubs.append(share)

print(f"same name, consecutive seasons ({len(same_name)}):   min {min(same_name):.2f} | median {np.median(same_name):.2f} | max {max(same_name):.2f}")
print(f"different clubs, consecutive seasons ({len(different_clubs)}): median {np.median(different_clubs):.2f} | 95th percentile {np.percentile(different_clubs, 95):.2f} | max {max(different_clubs):.2f}")
ambiguous.round({"squad overlap": 2})

same name, consecutive seasons (145):   min 0.09 | median 0.58 | max 0.93
different clubs, consecutive seasons (1229): median 0.00 | 95th percentile 0.11 | max 0.36


,earlier name,later name,earlier ends,later begins,ever share a season,squad overlap
0,Deccan Chargers,Sunrisers Hyderabad,2012,2013,False,0.50
1,Delhi Daredevils,Delhi Capitals,2018,2019,False,0.50
2,Kings XI Punjab,Punjab Kings,2020,2021,False,0.70
3,Royal Challengers Bangalore,Royal Challengers Bengaluru,2023,2024,False,0.62
4,Rising Pune Supergiants,Rising Pune Supergiant,2016,2017,False,0.39


**Reading B5**
- **The data show continuity for all five pairs.** Squad overlap is 0.39–0.70 for every ambiguous pair — inside the range of the same name in consecutive seasons (0.09–0.93, median 0.58) and above the highest overlap between any two different clubs (0.36; median 0.00).
- **The data cannot separate the replacement from the renames.** Deccan Chargers → Sunrisers Hyderabad (0.50) has exactly the overlap of Delhi Daredevils → Delhi Capitals (0.50) and more than Rising Pune Supergiants → Supergiant (0.39). A new club that took over much of the old squad looks like a renamed club in these data.

**Decisions on the ambiguous pairs.** *Data evidence* is what the cells above compute; *outside knowledge* is my recollection of IPL history, not checked against a source in this notebook, and should be verified before it is relied on.

| Pair | Data evidence | Outside knowledge | Decision |
|---|---|---|---|
| Delhi Daredevils → Delhi Capitals | never share a season; 2018 → 2019; squad overlap 0.50 | the same club rebranded ahead of the 2019 season | **rename** — one franchise |
| Kings XI Punjab → Punjab Kings | 2020 → 2021; overlap 0.70 | the same club rebranded in 2021 | **rename** |
| Royal Challengers Bangalore → Royal Challengers Bengaluru | 2023 → 2024; overlap 0.62 | the same club adopted the city's current spelling in 2024 | **rename** |
| Rising Pune Supergiants → Rising Pune Supergiant | 2016 → 2017; overlap 0.39; names one letter apart; the club exists only in these two seasons | the same club; only the spelling differs between its two seasons | **rename** (spelling variant) |
| Deccan Chargers → Sunrisers Hyderabad | 2012 → 2013; overlap 0.50 — in the data this looks like a rename | the Deccan Chargers franchise was terminated in 2012 and a new franchise took its place in Hyderabad for 2013 | **replacement** — two franchises. Decided on outside knowledge alone. |
| Pune Warriors / Rising Pune Supergiant(s) | gap of two seasons (2014, 2015) | different clubs in the same city | two franchises |
| Gujarat Lions / Gujarat Titans | gap of four seasons | different clubs in the same state | two franchises |

Result: 19 raw strings, **15 franchises**. The franchise column holds the franchise's *current name*; a 2008 row of the Delhi franchise therefore says "Delhi Capitals" — a presentation choice, which is why the raw columns stay beside the key.

### B6 · `teams.csv`, written by hand from the list above

`data/reference/teams.csv` has one row per raw string and the franchise's **current name**, built by hand from the 19 strings ordered by first season. First and last seasons are *not* stored in the file; they are computed here to check it.

In [6]:
teams = reference.read_teams(REFERENCE / "teams.csv")
print(f"{len(teams)} raw strings -> {teams['franchise'].nunique()} franchises")
teams.join(spans[["first_year", "last_year"]], on="team")

19 raw strings -> 15 franchises


,team,franchise,first_year,last_year
0,Chennai Super Kings,Chennai Super Kings,2008,2026
1,Deccan Chargers,Deccan Chargers,2008,2012
2,Delhi Daredevils,Delhi Capitals,2008,2018
3,Kings XI Punjab,Punjab Kings,2008,2020
4,Kolkata Knight Riders,Kolkata Knight Riders,2008,2026
5,Mumbai Indians,Mumbai Indians,2008,2026
6,Rajasthan Royals,Rajasthan Royals,2008,2026
7,Royal Challengers Bangalore,Royal Challengers Bengaluru,2008,2023
8,Kochi Tuskers Kerala,Kochi Tuskers Kerala,2011,2011
9,Pune Warriors,Pune Warriors,2011,2013


### B7 · Prove the map

Every check is a rule with a violation count. All must be zero.

In [7]:
franchise_of = teams.set_index("team")["franchise"]
team_seasons["franchise"] = team_seasons["team"].map(franchise_of)           # .map is used on purpose: unmapped strings show up as NaN and get counted
match_franchise = pd.DataFrame({c: matches[c].map(franchise_of) for c in ["team1", "team2", "toss_winner", "winner", "eliminator"]})
innings_franchise = pd.DataFrame({c: innings_table[c].map(franchise_of) for c in ["batting_team", "bowling_team"]})

franchise_years = team_seasons.drop_duplicates(["team", "year"])

def names_follow_each_other(group):
    """One franchise's raw names, in season order, form consecutive blocks: no A-B-A, and a new name starts the season after the old one ends."""
    group = group.sort_values("year")
    block = (group["team"] != group["team"].shift()).cumsum()
    blocks = group.groupby(block).agg(team=("team", "first"), first=("year", "min"), last=("year", "max"))
    return blocks["team"].is_unique and bool((blocks["first"].to_numpy()[1:] == blocks["last"].to_numpy()[:-1] + 1).all())

rules = []
def rule(check, name, broken):
    rules.append({"check": check, "rule": name, "violations": int(np.asarray(broken).sum())})

rule("coverage", "a raw string in any of the nine columns is not in teams.csv", sum(len(found - set(teams["team"])) for found in names.values()))
rule("coverage", "teams.csv lists a raw string that no table uses", len(set(teams["team"]) - set(all_names)))
rule("many-to-one", "a raw string appears twice in teams.csv (the map must be a function)", teams["team"].duplicated().sum())
rule("many-to-one", "a franchise name is not itself a raw string", ~teams["franchise"].isin(teams["team"]))
rule("franchise", "a franchise has two raw names in one season", franchise_years.groupby(["franchise", "year"])["team"].nunique() > 1)
rule("franchise", "a franchise's raw names do not follow each other in consecutive seasons",
     [not names_follow_each_other(group) for _, group in franchise_years.groupby("franchise")])
rule("matches", "both sides of a match are one franchise", match_franchise["team1"] == match_franchise["team2"])
rule("innings", "the batting and bowling side of an innings are one franchise", innings_franchise["batting_team"] == innings_franchise["bowling_team"])
for col in ["toss_winner", "winner", "eliminator"]:
    rule("matches", f"{col} is not one of the match's two franchises",
         match_franchise[col].notna() & (match_franchise[col] != match_franchise["team1"]) & (match_franchise[col] != match_franchise["team2"]))

validation = pd.DataFrame(rules)
assert (validation["violations"] == 0).all(), f"violations:\n{validation[validation['violations'] > 0]}"
validation

,check,rule,violations
0,coverage,a raw string in any of the nine columns is not in teams.csv,0
1,coverage,teams.csv lists a raw string that no table uses,0
2,many-to-one,a raw string appears twice in teams.csv (the map must be...,0
3,many-to-one,a franchise name is not itself a raw string,0
4,franchise,a franchise has two raw names in one season,0
5,franchise,a franchise's raw names do not follow each other in cons...,0
6,matches,both sides of a match are one franchise,0
7,innings,the batting and bowling side of an innings are one franc...,0
8,matches,toss_winner is not one of the match's two franchises,0
9,matches,winner is not one of the match's two franchises,0


### B8 · League matches per franchise-season

Notebook 02 (D3) inferred that matches *with* a `match_number` are league matches and the unnumbered ones are play-offs. Within a season every franchise should play the same number of league matches, so count them per franchise-season and compare each count with the season's **most common** count (the mode). The inference is tested too: if the counts contradict it, that is a finding.

In [8]:
league_ids = matches.loc[matches["match_number"].notna(), "match_id"]
league = team_seasons[team_seasons["match_id"].isin(league_ids)]
counts = league.groupby(["year", "franchise"]).size().rename("league matches").reset_index()

def most_common(values):
    """The most common count in a season. A tie is reported below and resolved to the larger count."""
    freq = values.value_counts()
    return int(freq[freq == freq.max()].index.max())

by_year = counts.groupby("year")["league matches"]
tied = by_year.agg(lambda s: int((s.value_counts() == s.value_counts().max()).sum()) > 1)
counts["season baseline"] = counts["year"].map(by_year.agg(most_common))
counts["deviation"] = counts["league matches"] - counts["season baseline"]
deviating = counts[counts["deviation"] != 0]

print("seasons where the most common count is tied:", tied[tied].index.tolist(), "-> the larger count is used as the baseline")
print(f"{len(counts)} franchise-seasons | {len(deviating)} deviate | below the baseline: {int((deviating['deviation'] < 0).sum())} | above: {int((deviating['deviation'] > 0).sum())}")
deviating[["year", "franchise", "league matches", "season baseline", "deviation"]]

seasons where the most common count is tied: [2009] -> the larger count is used as the baseline
166 franchise-seasons | 24 deviate | below the baseline: 22 | above: 2


,year,franchise,league matches,season baseline,deviation
2,2008,Delhi Capitals,13,14,-1
3,2008,Kolkata Knight Riders,13,14,-1
8,2009,Chennai Super Kings,13,14,-1
11,2009,Kolkata Knight Riders,13,14,-1
12,2009,Mumbai Indians,13,14,-1
14,2009,Rajasthan Royals,13,14,-1
32,2011,Rajasthan Royals,13,14,-1
33,2011,Royal Challengers Bengaluru,13,14,-1
34,2012,Chennai Super Kings,15,16,-1
35,2012,Deccan Chargers,15,16,-1


In [9]:
# Relate the deviations to the numbering gaps of notebook 02 (D2): each missing league match should leave two franchises one match short.
rows = []
for year, group in matches.groupby(matches["start_date"].dt.year):
    numbers = group["match_number"].dropna().astype(int)
    gaps = len(set(range(1, numbers.max() + 1)) - set(numbers))
    deviation = counts.loc[counts["year"] == year, "deviation"]
    rows.append({"year": year, "numbering gaps": gaps, "2 x gaps": 2 * gaps,
                 "franchise-matches short": int(-deviation[deviation < 0].sum()), "franchise-matches over": int(deviation[deviation > 0].sum())})
reconciliation = pd.DataFrame(rows).set_index("year")
reconciliation["short minus 2 x gaps"] = reconciliation["franchise-matches short"] - reconciliation["2 x gaps"]
reconciliation[(reconciliation[["numbering gaps", "franchise-matches short", "franchise-matches over"]] > 0).any(axis=1)]

,numbering gaps,2 x gaps,franchise-matches short,franchise-matches over,short minus 2 x gaps
year,,,,,
2008,1,2,2,0,0
2009,2,4,4,0,0
2011,1,2,2,0,0
2012,2,4,4,0,0
2015,1,2,2,0,0
2017,1,2,2,0,0
2024,2,4,6,0,2
2025,0,0,2,2,2


In [10]:
# The matches behind the two seasons that do not reconcile. 2025: the franchises above the baseline, and who they played.
matches_f = matches.join(match_franchise[["team1", "team2"]].add_suffix("_franchise"))
over_2025 = counts[(counts["year"] == 2025) & (counts["deviation"] > 0)]["franchise"].tolist()
short_2025 = counts[(counts["year"] == 2025) & (counts["deviation"] < 0)]["franchise"].tolist()
match_size = innings_table.groupby("match_id")["deliveries"].sum().rename("deliveries")

def between(franchises, year):
    inside = matches_f["team1_franchise"].isin(franchises) & matches_f["team2_franchise"].isin(franchises) & (matches_f["start_date"].dt.year == year)
    return matches_f[inside].join(match_size, on="match_id")[["match_number", "start_date", "team1", "team2", "winner", "outcome", "deliveries"]]

print("above the baseline in 2025:", over_2025, "| below:", short_2025)
print("\nnumbered 2025 matches between the two franchises above the baseline:")
print(between(over_2025, 2025).to_string(index=False))
print("\nnumbered 2025 matches between the two franchises below the baseline:")
print(between(short_2025, 2025).to_string(index=False))

above the baseline in 2025: ['Delhi Capitals', 'Punjab Kings'] | below: ['Kolkata Knight Riders', 'Royal Challengers Bengaluru']

numbered 2025 matches between the two franchises above the baseline:
 match_number start_date        team1          team2         winner   outcome  deliveries
           58 2025-05-08 Punjab Kings Delhi Capitals            NaN no result          63
           66 2025-05-24 Punjab Kings Delhi Capitals Delhi Capitals       NaN         249

numbered 2025 matches between the two franchises below the baseline:
 match_number start_date                 team1                       team2                      winner outcome  deliveries
            1 2025-03-22 Kolkata Knight Riders Royal Challengers Bengaluru Royal Challengers Bengaluru     NaN         223


In [11]:
# 2024: where the numbering ends, compared with the other seasons of ten franchises.
ten = counts.groupby("year")["franchise"].nunique().loc[lambda s: s == 10].index
ends = matches[matches["start_date"].dt.year.isin(ten)].groupby(matches["start_date"].dt.year)["match_number"].agg(highest="max", numbered="count")
ends["unnumbered"] = matches[matches["start_date"].dt.year.isin(ten)].groupby(matches["start_date"].dt.year)["match_number"].apply(lambda s: s.isna().sum())
ends

,highest,numbered,unnumbered
start_date,,,
2011,70,69,4
2022,70,70,4
2023,70,70,4
2024,69,67,4
2025,70,70,4
2026,70,70,4


**Reading B8**
- **League counts are equal within a season for 142 of 166 franchise-seasons;** 24 deviate. In **2009** the most common count is tied (four franchises on 14, four on 13) and the larger count is used as the baseline. The data alone cannot settle that tie — but with 13 as the baseline four franchises would be *above* it and none below, which contradicts the two numbering gaps found in 2009 in notebook 02, so 14 is the reading the other evidence supports.
- **Deviations go almost one way:** 22 are below the baseline (a match short) and 2 above.
- **In six seasons the shortfalls equal the numbering gaps exactly.** For 2008, 2009, 2011, 2012, 2015 and 2017 the number of franchise-matches short is 2 × the numbering gaps of notebook 02 (D2): every missing league match leaves two franchises one match short. So those gaps are real matches missing from the data, and the inference "numbered = league match" holds there: the baselines (14 or 16 per franchise) are the lengths of full league schedules, and no franchise has more numbered matches than that — except in 2025.
- **2024: the franchise counts show a match the numbering cannot.** Six franchise-matches are short, the two numbering gaps explain four. The numbering stops at 69, while every other season with ten franchises (2011, 2022, 2023, 2025, 2026) runs to 70, so the likeliest account is a missing last league match (#70) — the blind spot named in notebook 02: a missing match at the end of the numbering leaves no gap. Gujarat Titans (2 short), Kolkata Knight Riders (2), Rajasthan Royals (1) and Sunrisers Hyderabad (1) fit three missing fixtures; saying *which* fixtures needs outside knowledge.
- **2025: two franchises over, two under, no numbering gap.** Delhi Capitals and Punjab Kings have 15 numbered league matches because they appear in two numbered matches against each other: #58 on 8 May, a no result after 63 deliveries, and #66 on 24 May, a completed match — the abandoned fixture kept its number and its replay received another. Kolkata Knight Riders and Royal Challengers Bengaluru have 13: they met once (#1, the opener), and no second match between them exists in the data. Why a second fixture is absent (a match with no ball bowled would not be in the file) is outside knowledge. Since the numbering is unbroken, 2025 passes the notebook-02 test while two franchises are a match short.
- So **a gap-free season can still leave a franchise a match short** (2025), and **a gap-free tail can hide a missing match** (2024). Missing play-off matches are invisible to this method too: they carry no number.

## Part C — Seasons

The season label is not always a year (`2007/08`, `2009/10`, `2020/21`). The year per label is **derived** from the matches' start dates, and the one-to-one correspondence between labels and years is asserted — not typed.

In [12]:
by_label = matches.groupby("season")["start_date"].agg(first="min", last="max")
by_label["first_year"], by_label["last_year"] = by_label["first"].dt.year, by_label["last"].dt.year
assert (by_label["first_year"] == by_label["last_year"]).all(), "a season label spans two calendar years"

seasons = (by_label[["first_year"]].rename(columns={"first_year": "season_year"}).astype("int64")
           .reset_index().sort_values("season_year", ignore_index=True))
assert seasons["season"].is_unique and seasons["season_year"].is_unique        # labels and years correspond one-to-one
assert len(seasons) == matches["season"].nunique()

seasons.to_csv(REFERENCE / "seasons.csv", index=False)
pd.testing.assert_frame_equal(reference.read_seasons(REFERENCE / "seasons.csv"), seasons)
print(f"{len(seasons)} labels <-> {seasons['season_year'].nunique()} years; saved {REFERENCE / 'seasons.csv'} (read_seasons round-trips)")
seasons[seasons["season"] != seasons["season_year"].astype(str)]       # the labels that are not simply the year

19 labels <-> 19 years; saved ../data/reference/seasons.csv (read_seasons round-trips)


,season,season_year
0,2007/08,2008
2,2009/10,2010
12,2020/21,2020


**Reading Part C**
- The year of each label is **derived**: every label's matches fall within one calendar year (asserted), and labels and years pair off one-to-one (asserted), so nothing in `seasons.csv` was typed.
- The three labels that are not simply the year map to 2008, 2010 and 2020 — what notebook 01 found by eye. `2021` runs from April to October (India, then the UAE) but is still one calendar year.
- `season_year` is the tournament's calendar year, the right axis for any trend over time; the raw `season` label stays beside it.

## Part D — Apply the keys

`reference.add_keys()` adds `<raw column>_franchise` for every raw team column a table has, and `season_year`. Raw columns stay untouched. An unmapped value raises, a missing value stays missing, and the join must not change the row count (`validate="many_to_one"` plus an explicit count check).

In [13]:
teams = reference.read_teams(REFERENCE / "teams.csv")
seasons = reference.read_seasons(REFERENCE / "seasons.csv")
matches_keyed = reference.add_keys(matches, teams, seasons)
innings_keyed = reference.add_keys(innings_table, teams, seasons, matches=matches)      # innings has no season column: the year comes from its match

for name, before, after in [("matches", matches, matches_keyed), ("innings", innings_table, innings_keyed)]:
    pd.testing.assert_frame_equal(after[before.columns], before)                          # every raw column unchanged, dtypes included
    new = [c for c in after.columns if c not in before.columns]
    print(f"{name}: {len(before):,} rows before, {len(after):,} after | raw columns unchanged | added: {new}")
matches_keyed.loc[matches_keyed["team1"] != matches_keyed["team1_franchise"], ["season", "season_year", "team1", "team1_franchise"]].drop_duplicates("team1").head(6)

matches: 1,243 rows before, 1,243 after | raw columns unchanged | added: ['season_year', 'team1_franchise', 'team2_franchise', 'toss_winner_franchise', 'winner_franchise', 'eliminator_franchise']
innings: 2,514 rows before, 2,514 after | raw columns unchanged | added: ['season_year', 'batting_team_franchise', 'bowling_team_franchise']


,season,season_year,team1,team1_franchise
0,2007/08,2008,Royal Challengers Bangalore,Royal Challengers Bengaluru
1,2007/08,2008,Kings XI Punjab,Punjab Kings
2,2007/08,2008,Delhi Daredevils,Delhi Capitals
532,2016,2016,Rising Pune Supergiants,Rising Pune Supergiant


In [14]:
# An unmapped value must raise; a missing value must stay missing.
for label, bad in [("a typo in a team name", matches.assign(team1=matches["team1"].where(matches.index != 0, "Mumbai Indian"))),
                   ("an unknown season label", matches.assign(season=matches["season"].where(matches.index != 0, "2030")))]:
    try:
        reference.add_keys(bad, teams, seasons)
    except ValueError as err:
        print(f"{label:<25} -> ValueError: {err}")
print(f"\nmissing stays missing: winner is blank in {int(matches['winner'].isna().sum())} matches, winner_franchise in {int(matches_keyed['winner_franchise'].isna().sum())}")

a typo in a team name     -> ValueError: team1: not in the reference table: ['Mumbai Indian']
an unknown season label   -> ValueError: season: not in the reference table: ['2030']

missing stays missing: winner is blank in 25 matches, winner_franchise in 25


### Do the keys change any earlier result?

They should not. Two earlier results, recomputed on the franchise keys: the plain-result scoreboard rule of notebooks 02 and 03 ("the winner is the team with the higher total"), and the number of matches with a recorded winner.

In [15]:
regular = innings_keyed[~innings_keyed["is_super_over"]].set_index(["match_id", "innings"])
first, second = regular.xs(1, level="innings"), regular.xs(2, level="innings")
both = matches_keyed.set_index("match_id").join(first[["runs", "batting_team", "batting_team_franchise"]].add_prefix("i1_")).join(second[["runs", "batting_team", "batting_team_franchise"]].add_prefix("i2_"))
plain = both[both["winner"].notna() & both["method"].isna()]
chaser_ahead = plain["i2_runs"] > plain["i1_runs"]

raw_rule = plain["winner"] != np.where(chaser_ahead, plain["i2_batting_team"], plain["i1_batting_team"])
key_rule = plain["winner_franchise"] != np.where(chaser_ahead, plain["i2_batting_team_franchise"], plain["i1_batting_team_franchise"])
print(f"plain results where the winner is not the higher total - raw strings: {int(raw_rule.sum())} | franchise keys: {int(key_rule.sum())}  (of {len(plain):,})")
print(f"matches with a recorded winner - raw: {int(matches['winner'].notna().sum()):,} | keyed: {int(matches_keyed['winner_franchise'].notna().sum()):,}")

plain results where the winner is not the higher total - raw strings: 0 | franchise keys: 0  (of 1,195)
matches with a recorded winner - raw: 1,218 | keyed: 1,218


In [16]:
# What the key buys: one row per franchise instead of one per raw string (matches played, with the raw names behind each).
played = pd.concat([matches_keyed[["match_id", "team1", "team1_franchise"]].set_axis(["match_id", "team", "franchise"], axis=1),
                    matches_keyed[["match_id", "team2", "team2_franchise"]].set_axis(["match_id", "team", "franchise"], axis=1)])
per_franchise = played.groupby("franchise").agg(matches=("match_id", "nunique"), raw_names=("team", lambda s: " / ".join(sorted(s.unique()))))
per_franchise.sort_values("matches", ascending=False)

,matches,raw_names
franchise,,
Mumbai Indians,291,Mumbai Indians
Royal Challengers Bengaluru,286,Royal Challengers Bangalore / Royal Challengers Bengaluru
Delhi Capitals,281,Delhi Capitals / Delhi Daredevils
Kolkata Knight Riders,278,Kolkata Knight Riders
Punjab Kings,278,Kings XI Punjab / Punjab Kings
Chennai Super Kings,266,Chennai Super Kings
Rajasthan Royals,251,Rajasthan Royals
Sunrisers Hyderabad,211,Sunrisers Hyderabad
Gujarat Titans,77,Gujarat Titans


**Reading Part D**
- **Grain unchanged:** matches 1,243 → 1,243 rows and innings 2,514 → 2,514; every raw column is identical after the join, dtypes included. Keys are named `<raw column>_franchise` and `season_year`, and sit next to the raw columns.
- **Strict where it should be:** a typo or an unknown season label raises with the offending value; a missing value stays missing (the 25 matches with no `winner` have no `winner_franchise`).
- **No earlier result changes:** the plain-result scoreboard rule gives 0 violations on raw strings and on franchise keys, and the number of matches with a recorded winner is the same.
- **What the key buys:** one row per franchise instead of one per string — Delhi Capitals 281 matches (161 as Delhi Daredevils + 120 as Delhi Capitals), Royal Challengers Bengaluru 286 (240 + 46), Punjab Kings 278 (190 + 88), Rising Pune Supergiant 30 (14 + 16).
- **Two things the key does not decide.** A current name applied to the past is a presentation choice, not a fact — keep the raw columns for era-correct labels. And defunct franchises (Deccan Chargers, Kochi Tuskers Kerala, Pune Warriors, Gujarat Lions, Rising Pune Supergiant) stay in; whether they belong in all-time comparisons is an analysis decision.

## What this changes in the earlier worklists

| Earlier item | Now |
|---|---|
| Notebook 01, worklist 1 — season labels need a year mapping | **Done.** `seasons.csv`, derived and asserted one-to-one; `season_year` is added as a key. |
| Notebook 01, worklist 2–4 — team renames, spelling variant, look-alikes | **Done.** `teams.csv`: 19 strings → 15 franchises. Three renames and one spelling variant are merged; Deccan Chargers / Sunrisers Hyderabad stay apart on outside knowledge (the data cannot tell them from a rename); the two same-city pairs with a gap stay apart. |
| Notebook 02, D2 — numbering gaps and their blind spot | **Confirmed and extended.** Franchise counts reconcile with the gaps in six seasons, expose a match the numbering cannot see (2024), and show a franchise a match short with no gap at all (2025). |
| Notebook 02, D3 — "unnumbered matches are the play-offs" | **Not contradicted.** The season baselines (14 or 16 league matches per franchise) are full league schedules; only the two 2025 franchises exceed them, and a replayed fixture explains that. |
| Still open | Venues and cities (next session); consistency of player names; which fixtures the missing matches were (outside knowledge). |